In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import classification_report
import re
import joblib
import string

In [2]:
fake = pd.read_csv('Fake.csv')
true = pd.read_csv('True.csv')

In [3]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [5]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [6]:
fake['class']=0
true['class']=1

In [7]:
data = pd.concat([fake,true],axis=0)

In [9]:
data.sample(8)

,title,text,subject,date,class
10654,"19 retired U.S. generals, admirals back Clinto...",(Reuters) - A group of 19 retired U.S. general...,politicsNews,"February 25, 2016",1
8189,Former California lawmaker sentenced for laund...,"SACRAMENTO, Calif. (Reuters) - A former Califo...",politicsNews,"September 13, 2016",1
2366,Tillerson tightens the reins at State Department,WASHINGTON (Reuters) - U.S. Secretary of State...,politicsNews,"August 1, 2017",1
16026,BOOM! HARVARD LAW DEMOCRAT ALAN DERSHOWITZ Des...,Democrat Alan Dershowitz dismissed a major arg...,Government News,"Jul 17, 2017",0
12956,THIS MAY BE THE CLINTON’S Most DEPLORABLE Act ...,Scum of the earth isn t strong enough for thes...,politics,"Sep 20, 2016",0
6003,White House says looks to reschedule meeting w...,PHILADELPHIA (Reuters) - The White House said ...,politicsNews,"January 26, 2017",1
16890,IS OBAMA’S RADICAL AGENDA Responsible For Stat...,President Barack Obama has called the fight ag...,Government News,"Dec 26, 2015",0
11023,House Republicans to push Puerto Rico bill by ...,WASHINGTON (Reuters) - Republicans plan to bri...,politicsNews,"February 1, 2016",1


In [10]:
data = data.drop(['title','subject','date'],axis=1)

In [11]:
data.reset_index(inplace=True)

In [13]:
data.sample(5)

,index,text,class
20251,20251,If we had a real President who wasn t spending...,0
32002,8521,"WINDHAM, N.H. (Reuters) - Republican Donald Tr...",1
40715,17234,NAIROBI (Reuters) - Kenya s opposition leader ...,1
11514,11514,The cash flows through the State Department an...,0
36764,13283,"HANOVER, Germany (Reuters) - Members of the an...",1


In [15]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

In [16]:
data['text'] = data['text'].fillna('')

In [17]:
x = data['text']
y = data['class'] 

xtrain,xtest,ytrain,ytest = train_test_split(x,y,test_size=0.25,random_state=42)

In [18]:
vectorixer = TfidfVectorizer()
xvtrain = vectorixer.fit_transform(xtrain)
xvtest = vectorixer.transform(xtest)

In [19]:
lr = LinearRegression()
lr.fit(xvtrain,ytrain)

LinearRegression()

In [20]:
prediction = lr.predict(xvtest)
lr.score(xvtest,ytest)

0.7670601184383666

In [21]:
print(classification_report(ytest,prediction.round()))

              precision    recall  f1-score   support

        -2.0       0.00      0.00      0.00         0
        -1.0       0.00      0.00      0.00         0
         0.0       0.96      0.95      0.96      5895
         1.0       0.97      0.94      0.95      5330
         2.0       0.00      0.00      0.00         0

    accuracy                           0.95     11225
   macro avg       0.38      0.38      0.38     11225
weighted avg       0.96      0.95      0.95     11225



c:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\Lenovo\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [24]:
joblib.dump(vectorixer,'vectorizer.jb')
joblib.dump(lr,'model.jb')


['model.jb']